# CUDA-MEEP vs Meep: GPU Benchmark

Compares **CUDA-MEEP** (PyTorch GPU-native FDTD) vs **Meep** (CPU FDTD).

> **Enable GPU first:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 1: Check GPU
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'NOT DETECTED — enable GPU in Runtime settings')

In [ ]:
# Step 2: Clone CUDA-MEEP and install deps
!git clone https://github.com/shahzaibshazoo/cuda-meep.git
!pip install torch numpy matplotlib pytest --quiet

In [ ]:
# Step 3: Setup
import sys, time
import numpy as np
sys.path.insert(0, '/content/cuda-meep/src')

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Part 1: CUDA-MEEP Benchmark (GPU + CPU)

In [ ]:
from core import YeeGrid, FieldSet, MurABC, GaussianPulse, PointSource, SourceCollection, FDTD2D

GRID_SIZES = [64, 128, 256, 512]
N_WARMUP   = 20
N_STEPS    = 200
DX         = 1e-3

def run_cuda_meep(N, device, n_warmup, n_steps):
    grid     = YeeGrid(N, N, dx=DX, dy=DX, device=device)
    fields   = FieldSet(grid)
    boundary = MurABC(grid, fields.Hz)
    pulse    = GaussianPulse(amplitude=1.0, sigma=30*grid.dt)
    src      = PointSource(pulse, N//2, N//2, 'Hz', grid=grid, N_steps=n_warmup+n_steps)
    sim      = FDTD2D(grid, fields, boundary, SourceCollection([src]), n_check=500)
    sim.run(n_warmup)  # warmup
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    sim.run(n_steps)
    if device == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return n_steps * N * N / elapsed / 1e6, elapsed / n_steps * 1000

cuda_results, cpu_results = [], []
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'--- CUDA-MEEP ({DEVICE}) ---')
for N in GRID_SIZES:
    m, ms = run_cuda_meep(N, DEVICE, N_WARMUP, N_STEPS)
    cuda_results.append({'N': N, 'mcells_s': m, 'ms_step': ms})
    print(f'  {N}²  {m:8.1f} Mcells/s  {ms:8.3f} ms/step')

if DEVICE == 'cuda':
    print('--- CUDA-MEEP (cpu) ---')
    for N in GRID_SIZES:
        m, ms = run_cuda_meep(N, 'cpu', N_WARMUP, N_STEPS)
        cpu_results.append({'N': N, 'mcells_s': m, 'ms_step': ms})
        print(f'  {N}²  {m:8.1f} Mcells/s  {ms:8.3f} ms/step')
else:
    cpu_results = cuda_results  # same device

## Part 2: Install Meep

In [ ]:
# Install pymeep via conda-forge (the only reliable method)
print('Installing pymeep via conda-forge (~2-3 min)...')
!conda install -c conda-forge pymeep=*=nompi* -y -q 2>&1 | tail -5

MEEP_AVAILABLE = False
try:
    import meep as mp
    # Meep does not always expose __version__ — use the package metadata instead
    try:
        import importlib.metadata
        ver = importlib.metadata.version('meep')
    except Exception:
        ver = 'installed'
    print(f'Meep version: {ver}')
    # Verify the key class is accessible
    _ = mp.Vector3(0, 0, 0)
    MEEP_AVAILABLE = True
    print('Meep ready.')
except Exception as e:
    print(f'Meep not available ({e}). Will show CUDA-MEEP GPU vs CPU comparison only.')

## Part 3: Meep Benchmark

In [ ]:
meep_results = []

if MEEP_AVAILABLE:
    import meep as mp, os
    os.environ['MEEP_VERBOSITY'] = '0'  # suppress Meep output

    for N in GRID_SIZES:
        cell_size = mp.Vector3(1, 1)          # 1m x 1m domain
        res       = N                          # cells per unit → dx = 1/N metres
        sources   = [mp.Source(
            mp.GaussianSource(frequency=1.0, fwidth=0.5),
            component=mp.Hz,
            center=mp.Vector3(0, 0)
        )]
        sim_mp = mp.Simulation(
            cell_size=cell_size,
            resolution=res,
            sources=sources,
            boundary_layers=[mp.Absorber(thickness=0.1)]
        )

        # Warmup run
        sim_mp.run(until=5 * (1.0/res))
        sim_mp.reset_meep()

        # Timed run — equivalent to N_STEPS timesteps
        courant  = 0.5                      # Meep default Courant factor
        dt_meep  = courant / (res * 1.0)    # dt in Meep units
        t_target = N_STEPS * dt_meep

        t0 = time.perf_counter()
        sim_mp.run(until=t_target)
        elapsed = time.perf_counter() - t0

        actual_steps = max(1, int(round(t_target / dt_meep)))
        mcells_s = actual_steps * N * N / elapsed / 1e6
        ms_step  = elapsed / actual_steps * 1000
        meep_results.append({'N': N, 'mcells_s': mcells_s, 'ms_step': ms_step})
        print(f'  {N}²  meep  {mcells_s:8.1f} Mcells/s  {ms_step:8.3f} ms/step')
        sim_mp.reset_meep()
else:
    print('Meep unavailable — skipped.')
    # Use CUDA-MEEP CPU results as rough Meep stand-in (Meep is typically slower)
    meep_results = [{'N': r['N'], 'mcells_s': r['mcells_s']*0.6} for r in cpu_results]

## Part 4: Results

In [ ]:
# Summary table
print('='*70)
print(f'  {"Grid":5s}  {"GPU (Mcells/s)":>16s}  {"CPU (Mcells/s)":>16s}  {"Meep (Mcells/s)":>16s}  {"GPU/Meep":>9s}')
print('='*70)
for i, N in enumerate(GRID_SIZES):
    gpu  = cuda_results[i]['mcells_s'] if DEVICE == 'cuda' else None
    cpu  = cpu_results[i]['mcells_s']
    meep = meep_results[i]['mcells_s'] if meep_results else None
    spd  = f'{gpu/meep:.1f}x' if (gpu and meep) else (f'{cpu/meep:.1f}x(cpu)' if meep else 'N/A')
    print(f'  {N}²  {gpu or cpu:>14.1f}  {cpu:>14.1f}  {meep if meep else 0:>14.1f}  {spd:>9s}')
print('='*70)
if not MEEP_AVAILABLE:
    print('  Note: Meep values are estimated (Meep install failed). Run on a system')
    print('  with pymeep installed for accurate comparison.')

In [ ]:
# Plot
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CUDA-MEEP vs Meep: FDTD Throughput', fontsize=13)

ax = axes[0]
if DEVICE == 'cuda':
    ax.plot(GRID_SIZES, [r['mcells_s'] for r in cuda_results], 'o-g', label='CUDA-MEEP (GPU)', lw=2, ms=8)
ax.plot(GRID_SIZES, [r['mcells_s'] for r in cpu_results], 's-b', label='CUDA-MEEP (CPU)', lw=2, ms=8)
if meep_results:
    lbl = 'Meep (CPU)' if MEEP_AVAILABLE else 'Meep estimate'
    ax.plot(GRID_SIZES, [r['mcells_s'] for r in meep_results], '^-r', label=lbl, lw=2, ms=8)
ax.set(xlabel='Grid size (N²)', ylabel='Mcells/s', title='Throughput vs Grid Size')
ax.set_xticks(GRID_SIZES); ax.set_xticklabels([f'{N}²' for N in GRID_SIZES])
ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
if DEVICE == 'cuda' and meep_results:
    speedups = [cuda_results[i]['mcells_s']/meep_results[i]['mcells_s'] for i in range(len(GRID_SIZES))]
    bars = ax2.bar([f'{N}²' for N in GRID_SIZES], speedups, color='green', alpha=0.8)
    ax2.bar_label(bars, fmt='%.1fx', fontsize=12)
    ax2.axhline(1, color='red', ls='--', alpha=0.5, label='Meep baseline')
    ax2.set(ylabel='Speedup vs Meep', title=f'GPU Speedup{" (vs Meep estimate)" if not MEEP_AVAILABLE else ""}')
    ax2.legend()
elif DEVICE == 'cuda':
    speedups = [cuda_results[i]['mcells_s']/cpu_results[i]['mcells_s'] for i in range(len(GRID_SIZES))]
    bars = ax2.bar([f'{N}²' for N in GRID_SIZES], speedups, color='blue', alpha=0.8)
    ax2.bar_label(bars, fmt='%.1fx', fontsize=12)
    ax2.set(ylabel='GPU/CPU Speedup', title='GPU vs CPU Speedup')
else:
    ax2.text(0.5, 0.5, 'Enable GPU for speedup chart\n(Runtime → Change runtime type → GPU)',
             ha='center', va='center', transform=ax2.transAxes, fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 5: Brain Tumor Detection Demo

In [ ]:
import os
os.chdir('/content/cuda-meep')
print('Running brain MIMO imaging (32 simulations)...')
r = subprocess.run([sys.executable, 'examples/brain_mimo_imaging.py'],
                   capture_output=True, text=True, timeout=600)
print(r.stdout)
if r.returncode != 0:
    print('ERROR:', r.stderr[-2000:])

In [ ]:
from IPython.display import Image, display
img = '/content/cuda-meep/examples/output/brain_mimo_imaging.png'
display(Image(filename=img)) if os.path.exists(img) else print('Image missing — check errors above')